# 06-1. 주요 변수 기술통계

이 노트북은 `working/analysis/nia_2024_analysis_total.csv`를 기준으로 주요 변수 기술통계표를 생성한다. 출력 형식은 `outputs/06_distribution_figures/tables/descriptive_core_variables.xlsx`와 동일한 구조를 따르되, `중앙값` 열을 추가한다.

In [ ]:
from __future__ import annotations

from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
from xml.sax.saxutils import escape
import math
import os

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

SAVE_OUTPUTS = True

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "code":
    BASE_DIR = BASE_DIR.parent

DATA_PATH = BASE_DIR / "working" / "analysis" / "nia_2024_analysis_total.csv"
OUTPUT_DIR = BASE_DIR / "outputs" / "06_distribution_figures" / "tables"
CSV_OUTPUT_PATH = OUTPUT_DIR / "descriptive_core_variables.csv"
XLSX_OUTPUT_PATH = OUTPUT_DIR / "descriptive_core_variables.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

print("BASE_DIR:", BASE_DIR)
print("DATA_PATH:", DATA_PATH)
print("CSV_OUTPUT_PATH:", CSV_OUTPUT_PATH)
print("XLSX_OUTPUT_PATH:", XLSX_OUTPUT_PATH)

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"분석 데이터 파일을 찾지 못했습니다: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, low_memory=False)
print("데이터 shape:", df.shape)

CORE_VARIABLE_SPECS = [
    {
        "구분": "메인 DV",
        "변수명": "프로세스 개선의 정도 (프로세스 개선 관련 정보화 효과 인식)",
        "실제 변수명": "effect_proc_improve",
        "변수 유형": "순서형",
        "비고": "값 라벨: 1=전혀 효과 없음; 2=효과 없음; 3=보통; 4=효과 있음; 5=매우 효과 있음",
    },
    {
        "구분": "강건성 DV",
        "변수명": "정보화 효과 평균",
        "실제 변수명": "effect_average",
        "변수 유형": "연속형",
        "비고": "",
    },
    {
        "구분": "핵심 IV",
        "변수명": "AI 활용 범위 (AI 이용 유형의 개수)",
        "실제 변수명": "ai_use_sum",
        "변수 유형": "카운트형",
        "비고": "카운트형 변수, 분산/평균 비율 계산",
    },
    {
        "구분": "조절변수 W2",
        "변수명": "디지털 성숙도 (Digital Maturity Index, DMI)\n- 기업이 활용하고 있는 디지털 시스템 및 기술 유형의 개수",
        "실제 변수명": "dmi",
        "변수 유형": "카운트형",
        "비고": "카운트형 변수, 분산/평균 비율 계산",
    },
    {
        "구분": "보조 통제/강건성 변수",
        "변수명": "정보화 투자 범위 (정보화 투자 종류 개수)",
        "실제 변수명": "it_invest_sum",
        "변수 유형": "카운트형",
        "비고": "카운트형 변수, 분산/평균 비율 계산",
    },
    {
        "구분": "통제변수",
        "변수명": "기업 규모",
        "실제 변수명": "firm_size",
        "변수 유형": "순서형/범주형",
        "비고": "값 라벨: 1=10-49명; 2=50-249명; 3=250-999명; 4=1000명 이상",
    },
    {
        "구분": "보조 AI 변수",
        "변수명": "AI 실제 업무 이용 여부",
        "실제 변수명": "ai_use",
        "변수 유형": "이항",
        "비고": "1의 비율",
    },
    {
        "구분": "조절변수 W1",
        "변수명": "정보화 전담 인력 보유 여부",
        "실제 변수명": "it_org_any",
        "변수 유형": "이항",
        "비고": "1의 비율",
    },
]

missing_variables = [spec["실제 변수명"] for spec in CORE_VARIABLE_SPECS if spec["실제 변수명"] not in df.columns]
if missing_variables:
    raise KeyError("기술통계 대상 변수가 데이터에 없습니다: " + ", ".join(missing_variables))

In [ ]:
def numeric_series(var_name: str) -> pd.Series:
    return pd.to_numeric(df[var_name], errors="coerce")


def variance_mean_ratio(series: pd.Series) -> float:
    valid = pd.to_numeric(series, errors="coerce").dropna()
    mean_value = valid.mean()
    if valid.empty or pd.isna(mean_value) or mean_value == 0:
        return np.nan
    return float(valid.var(ddof=1) / mean_value)


def round_or_nan(value: float, digits: int = 3):
    if pd.isna(value):
        return np.nan
    return round(float(value), digits)


def excel_column_name(index: int) -> str:
    letters = []
    while index > 0:
        index, remainder = divmod(index - 1, 26)
        letters.append(chr(65 + remainder))
    return "".join(reversed(letters))


def excel_inline_string(text: str) -> str:
    escaped = escape(text)
    preserve = ' xml:space="preserve"' if text != text.strip() else ''
    return f"<is><t{preserve}>{escaped}</t></is>"


def dataframe_to_sheet_xml(frame: pd.DataFrame) -> str:
    rows_xml = []
    header_cells = []
    for col_idx, column_name in enumerate(frame.columns, start=1):
        cell_ref = f"{excel_column_name(col_idx)}1"
        header_cells.append(f'<c r="{cell_ref}" t="inlineStr">{excel_inline_string(str(column_name))}</c>')
    rows_xml.append(f'<row r="1">{"".join(header_cells)}</row>')

    for row_idx, row in enumerate(frame.itertuples(index=False, name=None), start=2):
        cells = []
        for col_idx, value in enumerate(row, start=1):
            if pd.isna(value):
                continue
            cell_ref = f"{excel_column_name(col_idx)}{row_idx}"
            if isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool):
                numeric_value = float(value)
                if math.isfinite(numeric_value):
                    cells.append(f'<c r="{cell_ref}"><v>{numeric_value}</v></c>')
            else:
                cells.append(f'<c r="{cell_ref}" t="inlineStr">{excel_inline_string(str(value))}</c>')
        rows_xml.append(f'<row r="{row_idx}">{"".join(cells)}</row>')

    last_col = excel_column_name(len(frame.columns))
    last_row = len(frame) + 1
    dimension = f"A1:{last_col}{last_row}"
    return (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">'
        f'<dimension ref="{dimension}"/>'
        '<sheetViews><sheetView workbookViewId="0"/></sheetViews>'
        '<sheetFormatPr defaultRowHeight="15"/>'
        f'<sheetData>{"".join(rows_xml)}</sheetData>'
        '</worksheet>'
    )


def write_minimal_xlsx(frame: pd.DataFrame, path: Path, sheet_name: str = "Sheet1") -> None:
    workbook_xml = (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" '
        'xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships">'
        '<sheets>'
        f'<sheet name="{escape(sheet_name)}" sheetId="1" r:id="rId1"/>'
        '</sheets>'
        '</workbook>'
    )
    workbook_rels_xml = (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
        '<Relationship Id="rId1" '
        'Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" '
        'Target="worksheets/sheet1.xml"/>'
        '</Relationships>'
    )
    root_rels_xml = (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
        '<Relationship Id="rId1" '
        'Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" '
        'Target="xl/workbook.xml"/>'
        '</Relationships>'
    )
    content_types_xml = (
        '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
        '<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">'
        '<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>'
        '<Default Extension="xml" ContentType="application/xml"/>'
        '<Override PartName="/xl/workbook.xml" '
        'ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/>'
        '<Override PartName="/xl/worksheets/sheet1.xml" '
        'ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/>'
        '</Types>'
    )

    with ZipFile(path, "w", compression=ZIP_DEFLATED) as zf:
        zf.writestr("[Content_Types].xml", content_types_xml)
        zf.writestr("_rels/.rels", root_rels_xml)
        zf.writestr("xl/workbook.xml", workbook_xml)
        zf.writestr("xl/_rels/workbook.xml.rels", workbook_rels_xml)
        zf.writestr("xl/worksheets/sheet1.xml", dataframe_to_sheet_xml(frame))

In [ ]:
rows = []
for spec in CORE_VARIABLE_SPECS:
    series = numeric_series(spec["실제 변수명"])
    row = {
        "구분": spec["구분"],
        "변수명": spec["변수명"],
        "실제 변수명": spec["실제 변수명"],
        "변수 유형": spec["변수 유형"],
        "유효 N": int(series.notna().sum()),
        "결측 N": int(series.isna().sum()),
        "평균 또는 비율": series.mean(),
        "표준편차": series.std(),
        "최솟값": series.min(),
        "중앙값": series.median(),
        "최댓값": series.max(),
        "분산/평균 비율": variance_mean_ratio(series) if spec["변수 유형"] == "카운트형" else np.nan,
        "비고": spec["비고"],
    }
    rows.append(row)

descriptive_core_variables = pd.DataFrame(rows)

for column in ["평균 또는 비율", "표준편차", "최솟값", "중앙값", "최댓값", "분산/평균 비율"]:
    descriptive_core_variables[column] = descriptive_core_variables[column].map(round_or_nan)

display(descriptive_core_variables)
display(Markdown(
    "**Table. 주요 변수 기술통계 해석 메모**\n"
    f"- 기술통계 대상 변수는 {len(descriptive_core_variables):,}개다.\n"
    f"- 중앙값 열을 추가해 순서형 변수와 count형 변수의 중심 경향을 함께 확인할 수 있게 했다.\n"
    f"- 출력 파일은 `{CSV_OUTPUT_PATH.relative_to(BASE_DIR)}` 및 `{XLSX_OUTPUT_PATH.relative_to(BASE_DIR)}`에 저장한다."
))

In [ ]:
if SAVE_OUTPUTS:
    descriptive_core_variables.to_csv(CSV_OUTPUT_PATH, index=False, encoding="utf-8-sig")
    write_minimal_xlsx(descriptive_core_variables, XLSX_OUTPUT_PATH, sheet_name="descriptive_core_variables")

print("CSV 저장 완료:", CSV_OUTPUT_PATH)
print("XLSX 저장 완료:", XLSX_OUTPUT_PATH)
print(descriptive_core_variables.to_string(index=False))